# Lecture 26 - Model Evaluation and Cross-Validation

## Learning Objectives

- Interpret a confusion matrix and basic classification metrics
- Use classification_report() and confusion_matrix()
- Evaluate regression models with MSE, MAE, and R-squared
- Perform k-fold cross-validation with cross_val_score()
- Understand learning curves and the bias-variance trade-off

## Key Topics

- Confusion matrix, accuracy, precision, recall, F1-score
- classification_report() and confusion_matrix()
- Regression metrics: MSE, MAE, R-squared
- cross_val_score() and cross_validate()
- Learning curves and validation curves
- Bias-variance trade-off

## Confusion Matrix and Classification Metrics

A **confusion matrix** is a table that compares predicted labels against true labels. For binary classification it has four entries:

- **True Positives (TP)** — correctly predicted positives
- **True Negatives (TN)** — correctly predicted negatives
- **False Positives (FP)** — incorrectly predicted positives (Type I error)
- **False Negatives (FN)** — incorrectly predicted negatives (Type II error)

From these four numbers we derive the key classification metrics:
- **Accuracy**: (TP + TN) / (TP + TN + FP + FN) — overall correctness
- **Precision**: TP / (TP + FP) — how many selected items are relevant
- **Recall**: TP / (TP + FN) — how many relevant items are selected
- **F1-score**: harmonic mean of precision and recall — a balanced metric when classes are imbalanced

Scikit-learn's `classification_report()` prints all these metrics at once, making it easy to assess model performance at a glance.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
import seaborn as sns
import matplotlib.pyplot as plt

iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
knn = KNeighborsClassifier(n_neighbors=5).fit(X_tr, y_tr)
y_pred = knn.predict(X_te)

cm = confusion_matrix(y_te, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title("Confusion Matrix - Iris KNN")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.show()

In [ ]:
print(classification_report(y_te, y_pred, target_names=iris.target_names))

## Regression Metrics: MSE, MAE, R-squared

Regression problems require different evaluation metrics:

- **Mean Absolute Error (MAE)**: average absolute difference between predictions and true values. Interpretable in the original units.
- **Mean Squared Error (MSE)**: average squared difference. Penalises large errors more heavily.
- **R-squared (R²)**: proportion of variance in the target explained by the model. Ranges from -∞ to 1, with 1 being perfect.

MAE is more intuitive, while MSE is mathematically convenient (differentiable everywhere). R-squared gives you a scale-free measure of fit quality. Always look at all three to get a complete picture.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression

X_r, y_r = make_regression(n_samples=200, n_features=1, noise=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.3, random_state=42)

lr = LinearRegression().fit(X_tr, y_tr)
y_pred_r = lr.predict(X_te)

print(f"MAE:  {mean_absolute_error(y_te, y_pred_r):.2f}")
print(f"MSE:  {mean_squared_error(y_te, y_pred_r):.2f}")
print(f"RMSE: {mean_squared_error(y_te, y_pred_r, squared=False):.2f}")
print(f"R²:   {r2_score(y_te, y_pred_r):.3f}")

## Cross-Validation with cross_val_score()

A single train/test split can be noisy — your estimate of performance depends heavily on which points end up in the test set. **K-fold cross-validation** solves this by splitting the data into k folds, training on k-1 folds, and evaluating on the held-out fold. This process repeats k times, and the scores are averaged.

Scikit-learn's `cross_val_score()` makes this trivially easy. For more detail (fit times, training scores), use `cross_validate()` which returns a dictionary of metrics.

Cross-validation gives a more reliable estimate of how your model will generalise to unseen data and helps detect overfitting when training scores far exceed validation scores.

In [ ]:
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_digits

digits = load_digits()
X_d, y_d = digits.data, digits.target

rf = RandomForestClassifier(n_estimators=50, random_state=42)
scores = cross_val_score(rf, X_d, y_d, cv=5)
print(f"CV scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

In [ ]:
# Detailed cross-validation with training scores
cv_results = cross_validate(rf, X_d, y_d, cv=5,
                            return_train_score=True)
print("Test scores:", cv_results["test_score"])
print("Train scores:", cv_results["train_score"])
print(f"Mean train: {cv_results['train_score'].mean():.3f}  "
      f"Mean test: {cv_results['test_score'].mean():.3f}")

## Learning Curves and the Bias-Variance Trade-Off

A **learning curve** plots model performance on both the training and validation sets as a function of the number of training examples.

- If training score stays high but validation score stays low, the model is **overfitting** (high variance) — it memorises the training data but fails to generalise. Adding more data or reducing model complexity usually helps.
- If both scores converge at a low level, the model is **underfitting** (high bias) — it is too simple to capture the underlying pattern. A more complex model or better features is needed.

This is the **bias-variance trade-off**: simple models have high bias but low variance; complex models have low bias but high variance. The goal is to find the sweet spot that minimises total error on unseen data.

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np

train_sizes, train_scores, test_scores = learning_curve(
    RandomForestClassifier(n_estimators=50, random_state=42),
    X_d, y_d, cv=5, train_sizes=np.linspace(0.1, 1.0, 10),
    scoring="accuracy"
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure(figsize=(8, 4))
plt.plot(train_sizes, train_mean, "o-", label="Training score")
plt.plot(train_sizes, test_mean, "o-", label="Validation score")
plt.xlabel("Training examples")
plt.ylabel("Accuracy")
plt.title("Learning Curve - Random Forest on Digits")
plt.legend()
plt.grid(True)
plt.show()

## Data Science Connection

Evaluation is where machine learning meets scientific rigour. A model is only as good as its measured performance on unseen data. Cross-validation gives you honest estimates, confusion matrices diagnose where your classifier goes wrong, and learning curves guide you toward collecting more data or adjusting model complexity. Every Kaggle competition winner and every production ML system relies on these evaluation tools to build trustworthy models.